In [ ]:
import re
# ^^^ pyforest auto-imports - don't write above this line
import pandas as pd
pd.options.display.max_columns = None
pd.options.plotting.backend = 'plotly'

import moccalib as ml

In [ ]:
import re
# ^^^ pyforest auto-imports - don't write above this line
import sqlalchemy as db
engine = db.create_engine('sqlite:///snapshot.db')
with engine.connect() as conn:
    print(conn.execute(db.text('select count(*) from snapshot')).fetchall())

In [ ]:
import sqlalchemy as db
engine = db.create_engine('sqlite:///snapshot.db')
with engine.connect() as conn:
    df = pd.read_sql('select * from snapshot where ik1==14 and ik2==14 limit 1000', con=conn)

In [ ]:
df.a.describe()

In [ ]:
# !scp camk:/work/chuck/mocca/gwiktoro/devel_bse_Albrecht/src/run/test_MT_model1/xrb.dat .
!scp camk:/work/chuck/mocca/gwiktoro/devel_bse_Albrecht/src/run/test_MT_model1_dyn/xrb.dat xrb_dyn.dat

In [ ]:
# reformats the write statement from the code to column names for reading with pandas
write_stm = """idbin, id1, id2, j1, j2, tphys, dtm,                                                                                                     & age, epoch(1),                                                                                                                                         
     & epoch(2), kstar(1), kstar(2), mass(1), mass(2), sep, ecc,                                                                                              
     &       rad(1), rad(2), lumin(1), lumin(2), massc(1), massc(2),                                                                                          
     &       radc(1), radc(2), menv(1), menv(2), renv(1), renv(2),                                                                                            
     &       ospin(1), ospin(2), dmt(1), dmt(2), dmr(1), dmr(2), rol(1),                                                                                      
     &       rol(2),dmdt(1), dmdt(2), dm1, dm2 , tb, Lx
"""
names = (re.sub("[^(\w]\d+[^)]", "", write_stm)  # remove line numbers
         .replace('&', '')  # remove continuation marks
         .replace(',', ' ')  # use only whitespaces as delimiters
         .replace('(','_')
         .replace(')','')
         .split()
        )
names

In [ ]:
df = pd.read_csv('xrb.dat', names=names, sep='\s+')
df

## Time steps consistency

In [ ]:
df.idbin.unique()

Problem with 78724, 78712
- everytime we pass through the standev step (1 Myr), e.g. tphys==5.0, there is a mishap in dtm and tphys.diff
- evolv2b.f l:1019 is probably responsible 
- use loc for better visibility!

Problem with 78660
- negative dtm

Problem with 107114, 78204
- seems to be different systems

In [ ]:
idbins = [78724, 78712]
idbins = [107114, 78204]
for idbin in idbins:  # df.idbin.unique():
    print(f"{idbin=}")
    dfi = df.query(f'idbin=={idbin}').copy()#.loc[lambda x: x.tphys.between(4.95, 5.05)]#.reset_index()

    dfi['dtphys']=dfi.tphys.diff()
    display(dfi[['tphys','dtm','dtphys','kstar_1','kstar_2','sep']])

    display(dfi[['tphys','dtm','dtphys','kstar_1','kstar_2','sep']].query('abs(dtm-dtphys)>0.00001'))

## Mass transfer rates 

In [ ]:
df['Mdot'] = df['dm1']/df['tb']

In [ ]:
display(df.Mdot.describe())
# df[df.idbin!=77748].Mdot.apply(np.log10).hist(nbins=40, log_y=True)
df.Mdot.apply(np.log10).hist(nbins=40, log_y=True)

- negative values (probably because RLOF was calculated for non-RLfilling systems
- peak at 4e-8 (for older systems)


In [ ]:
df[df.Mdot.apply(np.log10).between(-7.5, -7)].idbin.value_counts()

In [ ]:
df.query('idbin==77748')

## X-ray luminosity

In [ ]:
display(df.Lx.describe())
df.Lx.apply(np.log10).hist(nbins=40, log_y=True)

In [ ]:
df.plot(kind='scatter', x='dtm', y='Lx', log_y=True)

- negative values
- good range of values
- reversed(?) relation for Lx(dtm)

In [ ]:
df['mass_don'] = df.mass_2.where(df.j1==2, df.mass_1)
df['rad_don'] = df.rad_2.where(df.j1==2, df.rad_1)
df['rol_don'] = df.rol_2.where(df.j1==2, df.rol_1)
df['kstar_don'] = df.kstar_2.where(df.j1==2, df.kstar_1)

df['mass_acc'] = df.mass_2.where(df.j1==1, df.mass_1)
df['rad_acc'] = df.rad_2.where(df.j1==1, df.rad_1)
df['rol_acc'] = df.rol_2.where(df.j1==1, df.rol_1)
df['kstar_acc'] = df.kstar_2.where(df.j1==1, df.kstar_1)

In [ ]:
df.query('Lx>0').plot(kind='scatter', x='tphys', y='Lx', log_y=True, size='dtm', color='kstar_acc', opacity=0.5).update_traces(marker_line_width=0)

### Dynamical simulations

In [ ]:
dfd = pd.read_csv('xrb_dyn.dat', names=names, sep='\s+')
dfd

In [ ]:
dfd.tphys.describe()

In [ ]:
for idbin in [78666]:  # dfd.idbin.unique():
    print(f"{idbin=}")
    dfi = dfd.query(f'idbin=={idbin}').copy()#.loc[lambda x: x.tphys.between(4.95, 5.05)]#.reset_index()

    dfi['dtphys']=dfi.tphys.diff()
    display(dfi[['tphys','dtm','dtphys','kstar_1','kstar_2','sep']])

    display(dfi[['tphys','dtm','dtphys','kstar_1','kstar_2','sep']].query('abs(dtm-dtphys)>0.00001'))

In [ ]:
a=None
f"{a}"

In [ ]:
mdon = 16.642224
Rdon = 4.493982
Rldon = 4.538849
3e-6 * min(mdon, 5)**2 * np.log(Rdon/Rldon)**3

In [ ]:
(df.rad_don / df.rol_don).describe()

In [ ]:
df.Lx.describe()

In [ ]:
df['Lx'] = df.apply(lambda row: Lxacc(row.mass_acc, row.rad_acc, row.dm1, row.kstar_acc), axis=1)

In [ ]:
with pd.option_context('display.max_columns', None):
    display(dff)

In [ ]:
dff['Lx'] = dff.apply(lambda row: 1.2e35 * row[f'mass({int(row.j2)})'] * row[f'mdot'] * 1e6 / row[f'rad({int(row.j2)})'], axis=1)

In [ ]:
dff.Lx.describe()

In [ ]:
df[(df.idbin != df.id1)]

In [ ]:
(df.tphys - df.age - df['epoch(2)']).agg(['min','max'])

In [ ]:
df.j2.unique()

In [ ]:
df[['dmdt(1)', 'dmdt(2)']].describe()

In [ ]:
# Mass tranfser rate for donors (j1==2)
df.query('j1==2')['dmdt(2)'].describe()#.apply(np.log10).hist()

In [ ]:
df.query('j1==1')['dmdt(1)'].describe()

In [ ]:
df[(df['j1'] == 1) & (df['dmdt(1)'] < 0)]

In [ ]:
df[(df['j1'] == 2) & (df['dmdt(2)'] < 0)]

In [ ]:
df[(df['dmdt(1)']<0) & (df['dmdt(2)'] < 0)]

In [ ]:
id1 = 78736
df.query('id1==@id1 or id2==@id1')

In [ ]:
snapshot = ml.read_snapshot('snapshot.dat')
snapshot

In [ ]:
snapshot.query("idd1==@id1 or idd2==@id1")

In [ ]:
from pathlib import Path

In [ ]:
datapath = Path('./zzz')
if not datapath.exists():
    print("FILE DO NOT EXIST!!!")

In [ ]:
import re

In [ ]:
re.findall("\line.replace(' ','')

In [ ]:
def row_to_dict(line):
    row = {}
    for k, v in map(lambda x: x.split('='), re.findall(rf'\w+=[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eEdD][+-]?\d+)?', line.replace(' ',''))):
        if v.isnumeric():
            row[k]=int(v)
        else:
            try: 
                row[k]=float(v)
            except:  # is actually string (object)
                row[k]=v

    return row

def read_zzz(path, row_selector=None, row_transformer=None, return_dataframe=True):
    data = []
    with open(datapath, 'r') as f:
        while line := f.readline():
            if row_selector is not None and not row_selector(line):
                continue
            if row_transformer is not None:
                line = row_to_dict(line)
            data.append(line)
    if return_dataframe:
        return pd.DataFrame(data)
    else:
        return data
    
read_zzz(datapath, row_selector = lambda line: line.startswith(' LH3:'), row_transformer=row_to_dict)

In [ ]:
import seaborn as sns
sns.lineplot(data=df, x='tphys', y='B')

In [ ]:
1.9

In [ ]:
df = read_zzz(datapath,
              row_selector = lambda line: line.startswith(' LH1:'), 
              row_transformer=row_to_dict
             )
df

In [ ]:
df.tphys-df.epoch-df.tacc

In [ ]:
(df.k==13).sum()

In [ ]:
import seaborn as sns
sns.lineplot(data=df, x='tphys', y='B')

In [ ]:
df.groupby(['id','k']).plot(x='tphys',y='B')